# MAVIS — Pipeline de Embeddings Multimodales (FakeVV Test Set)


## 0. Setup y configuración


In [ ]:
from google import genai
from google.genai import types
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import sqlite3
import json
import time
import os
from dotenv import load_dotenv
from faster_whisper import WhisperModel

load_dotenv()
API_KEY = os.getenv('GEMINI_API_KEY')

MODEL_NAME = 'gemini-embedding-2-preview'
OUTPUT_DIMENSIONS = 3072
DELAY_BETWEEN_REQUESTS = 10  # segundos entre peticiones (ajustar según tier)
DB_PATH = 'mavis_cache.db'
TEST_INDEX_PATH = 'test_index.json'

os.environ['GEMINI_API_KEY'] = API_KEY
client = genai.Client()

os.makedirs('results/figures', exist_ok=True)

# Whisper model for transcript extraction
try:
    # Intentar usar GPU (tarjeta gráfica) si está disponible
    whisper_model = WhisperModel('small', device='cuda', compute_type='float16')
    print('🎧 Whisper cargado en GPU (cuda).')
except Exception:
    # Fallback automático a CPU
    whisper_model = WhisperModel('small', device='cpu', compute_type='int8')
    print('🎧 Whisper cargado en CPU (int8).')

print('✅ Setup completado (incl. Whisper para transcripts).')


## 1. Cache SQLite


In [ ]:
def init_db(db_path=DB_PATH):
    conn = sqlite3.connect(db_path)
    conn.execute('''
        CREATE TABLE IF NOT EXISTS embeddings (
            raw_id     TEXT,
            modality   TEXT,
            vector     BLOB,
            created_at TEXT DEFAULT CURRENT_TIMESTAMP,
            PRIMARY KEY (raw_id, modality)
        )
    ''')
    conn.commit()
    return conn


def save_embedding(conn, raw_id, modality, emb):
    '''Guarda un embedding en la DB. Si ya existe, no sobreescribe.'''
    conn.execute(
        'INSERT OR IGNORE INTO embeddings (raw_id, modality, vector) VALUES (?, ?, ?)',
        (raw_id, modality, emb.astype(np.float32).tobytes())
    )
    conn.commit()


def load_embedding(conn, raw_id, modality):
    '''Carga un embedding de la DB. Devuelve None si no existe.'''
    row = conn.execute(
        'SELECT vector FROM embeddings WHERE raw_id=? AND modality=?',
        (raw_id, modality)
    ).fetchone()
    if row is None:
        return None
    return np.frombuffer(row[0], dtype=np.float32)


def get_or_compute(conn, raw_id, modality, compute_fn):
    '''Lee de cache; si no existe, calcula, guarda y devuelve.'''
    emb = load_embedding(conn, raw_id, modality)
    if emb is not None:
        return emb
    emb = compute_fn()
    save_embedding(conn, raw_id, modality, emb)
    return emb


def cache_progress(conn):
    '''Muestra cuántos embeddings hay por modalidad.'''
    rows = conn.execute(
        'SELECT modality, COUNT(*) FROM embeddings GROUP BY modality'
    ).fetchall()
    return {mod: count for mod, count in rows}


db = init_db()
print('✅ Cache SQLite inicializada:', DB_PATH)
print('   Progreso actual:', cache_progress(db))


## 2. Funciones de embedding (API)


In [ ]:
import subprocess
import json as json
import os
def get_text_embedding(text, task_prefix='task: fact checking | query: '):
    full_text = f'{task_prefix}{text}'
    response = client.models.embed_content(
        model=MODEL_NAME,
        contents=full_text,
        config=types.EmbedContentConfig(output_dimensionality=OUTPUT_DIMENSIONS)
    )
    emb = np.array(response.embeddings[0].values)
    return emb / np.linalg.norm(emb)


def get_video_embedding(file_path, mime_type='video/mp4', max_duration=120):
    # Check duration before uploading to avoid API rejection
    probe = subprocess.run(
        ["ffprobe", "-v", "quiet", "-print_format", "json", "-show_streams", file_path],
        capture_output=True, text=True
    )
    streams = json.loads(probe.stdout).get("streams", [])
    with open(file_path, 'rb') as f:
        file_bytes = f.read()
    response = client.models.embed_content(
        model=MODEL_NAME,
        contents=[types.Part.from_bytes(data=file_bytes, mime_type=mime_type)],
        config=types.EmbedContentConfig(output_dimensionality=OUTPUT_DIMENSIONS)
    )
    emb = np.array(response.embeddings[0].values)
    return emb / np.linalg.norm(emb)


def get_grouped_frames_embedding(frame_paths):
    '''Embedding unificado a partir de múltiples frames.'''
    parts = []
    for fp in frame_paths:
        with open(fp, 'rb') as f:
            parts.append(types.Part.from_bytes(data=f.read(), mime_type='image/png'))
    response = client.models.embed_content(
        model=MODEL_NAME,
        contents=[types.Content(parts=parts)],
        config=types.EmbedContentConfig(output_dimensionality=OUTPUT_DIMENSIONS)
    )
    emb = np.array(response.embeddings[0].values)
    return emb / np.linalg.norm(emb)




def get_transcript(file_path):
    '''Extrae el transcript de audio de un vídeo usando Whisper.'''
    # Check duration
    probe = subprocess.run(
        ["ffprobe", "-v", "quiet", "-print_format", "json", "-show_streams", file_path],
        capture_output=True, text=True
    )
    streams = json.loads(probe.stdout).get("streams", [])
    has_audio = any(s.get("codec_type") == "audio" for s in streams)
    if not has_audio:
        raise ValueError(f"No audio track in {os.path.basename(file_path)}")
    
    segments, info = whisper_model.transcribe(file_path, language="en")
    text = " ".join(seg.text.strip() for seg in segments)
    return text.strip()


def get_transcript_embedding(file_path):
    '''Extrae transcript y genera su embedding de texto.'''
    transcript = get_transcript(file_path)
    if not transcript:
        raise ValueError(f"Transcript vacío para {os.path.basename(file_path)}")
    return get_text_embedding(transcript, task_prefix='task: fact checking | document: ')

print('✅ Funciones de embedding definidas.')


## 3. Rate limiter


In [ ]:
import collections
import threading

class RateLimiter:
    '''
    Proactively tracks RPM and RPD to avoid hitting free-tier quota.
    Pauses before a request when approaching the RPM ceiling.
    Raises immediately when RPD daily budget is exhausted.
    '''
    def __init__(self, rpm_limit=100, rpd_limit=1000, rpm_headroom=0.85, rpd_headroom=0.92):
        self.rpm_limit   = rpm_limit
        self.rpd_limit   = rpd_limit
        self.rpm_budget  = int(rpm_limit * rpm_headroom)   # 85 safe RPM
        self.rpd_budget  = int(rpd_limit * rpd_headroom)   # 920 safe RPD
        self._timestamps = collections.deque()
        self._day_count  = 0
        self._lock       = threading.Lock()

    def _purge(self):
        now = time.time()
        while self._timestamps and now - self._timestamps[0] > 60:
            self._timestamps.popleft()

    def wait_if_needed(self):
        with self._lock:
            if self._day_count >= self.rpd_budget:
                raise Exception(
                    f'RPD budget exhausted ({self._day_count}/{self.rpd_budget}). '
                    'Wait for daily reset (midnight Pacific Time).'
                )
            self._purge()
            rpm_now = len(self._timestamps)
            if rpm_now >= self.rpm_budget:
                wait_s = 60 - (time.time() - self._timestamps[0]) + 0.5
                if wait_s > 0:
                    print(f'\n⏳ RPM={rpm_now}/{self.rpm_budget}. Waiting {wait_s:.1f}s...', end='')
                    time.sleep(wait_s)
                    self._purge()

    def record(self, n=1):
        with self._lock:
            now = time.time()
            for _ in range(n):
                self._timestamps.append(now)
            self._day_count += n

    def status(self):
        self._purge()
        return (
            f'RPM {len(self._timestamps)}/{self.rpm_budget} | '
            f'RPD {self._day_count}/{self.rpd_budget} '
            f'(remaining: {self.rpd_budget - self._day_count})'
        )


rate_limiter = RateLimiter(rpm_limit=100, rpd_limit=1000)


def safe_embed_fetch(content_func, *args, max_retries=5, base_delay=30, **kwargs):
    '''Calls the API with exponential backoff on 429. Records 1 RPD per call.'''
    rate_limiter.wait_if_needed()
    for attempt in range(max_retries):
        try:
            result = content_func(*args, **kwargs)
            rate_limiter.record(1)
            return result
        except Exception as e:
            msg = str(e).lower()
            if '429' in msg or 'quota' in msg or 'too many requests' in msg:
                wait_time = base_delay * (1.5 ** attempt)
                print(f'⏳ Rate limit. Waiting {int(wait_time)}s... (attempt {attempt+1}/{max_retries})')
                time.sleep(wait_time)
            else:
                print(f'❌ Error: {e}')
                raise
    raise Exception('❌ Max retries exceeded.')


print(f'✅ Smart rate limiter ready (RPM_budget={rate_limiter.rpm_budget}, RPD_budget={rate_limiter.rpd_budget}).')


In [ ]:
def get_text_embeddings_batch(items, conn, batch_size=50):
    """
    Batch-embed a list of (raw_id, modality, full_text) tuples.

    Uses client.models.embed_content with contents=[Content, Content, ...]
    which returns one embedding per Content (SDK 1.72 batch behaviour).
    One API call per batch  ->  1 RPD per batch (not 1 RPD per text).

    Crash-safe: saves each batch to SQLite immediately after the API call
    succeeds. A crash between batches loses at most ONE in-flight batch.
    INSERT OR IGNORE makes re-runs always safe.

    `full_text` must already include the task prefix.
    Returns the total number of embeddings saved.
    """
    saved = 0
    total_batches = (len(items) + batch_size - 1) // batch_size

    for b_idx, start in enumerate(range(0, len(items), batch_size)):
        batch = items[start:start + batch_size]

        # Build one Content object per text (NOT a flat list of strings,
        # which would be treated as a single multi-part content).
        contents = [
            types.Content(parts=[types.Part(text=full_text)])
            for _, _, full_text in batch
        ]

        rate_limiter.wait_if_needed()

        for attempt in range(5):
            try:
                response = client.models.embed_content(
                    model=MODEL_NAME,
                    contents=contents,
                    config=types.EmbedContentConfig(output_dimensionality=OUTPUT_DIMENSIONS)
                )
                rate_limiter.record(1)  # 1 RPD for the whole batch
                break
            except Exception as e:
                msg = str(e).lower()
                if '429' in msg or 'quota' in msg or 'too many requests' in msg:
                    wait_s = 30 * (1.5 ** attempt)
                    print(f'\n⏳ Batch rate-limit. Waiting {int(wait_s)}s... (attempt {attempt+1}/5)')
                    time.sleep(wait_s)
                else:
                    print(f'\n❌ Batch error: {e}')
                    raise
        else:
            raise Exception('❌ Max batch retries exceeded.')

        # Save immediately — crash after this point loses only the NEXT batch
        for (raw_id, modality, _), emb_obj in zip(batch, response.embeddings):
            emb = np.array(emb_obj.values, dtype=np.float32)
            emb = emb / np.linalg.norm(emb)
            save_embedding(conn, raw_id, modality, emb)
            saved += 1

        print(f'\r  Batch [{b_idx+1}/{total_batches}] | saved {saved} embs | {rate_limiter.status()}', end='', flush=True)

    print()
    return saved


print('✅ get_text_embeddings_batch() defined (SDK 1.72 compatible, crash-safe).')


In [ ]:
# ── Validation: test embed_content batch (3 Content objects) ─────────────
# SDK 1.72: pass contents=[Content(...), Content(...), ...]
# Returns one embedding per Content. Consumes 1 RPD.

_test_texts = [
    'task: fact checking | query: Breaking news: president announces new policy',
    'task: fact checking | query: Fake headline: scientist discovers time travel',
    'task: fact checking | document: The president held a press conference today.',
]
_test_contents = [
    types.Content(parts=[types.Part(text=t)])
    for t in _test_texts
]

try:
    rate_limiter.wait_if_needed()
    _val = client.models.embed_content(
        model=MODEL_NAME,
        contents=_test_contents,
        config=types.EmbedContentConfig(output_dimensionality=OUTPUT_DIMENSIONS)
    )
    rate_limiter.record(1)
    assert len(_val.embeddings) == len(_test_texts), (
        f'Expected {len(_test_texts)} embeddings, got {len(_val.embeddings)}'
    )
    for i, emb_obj in enumerate(_val.embeddings):
        v = np.array(emb_obj.values)
        assert len(v) == OUTPUT_DIMENSIONS, f'Expected dim {OUTPUT_DIMENSIONS}, got {len(v)}'
        print(f'  text[{i}] → dim={len(v)}, norm={np.linalg.norm(v):.4f} ✅')
    print(f'\n✅ Batch embed works ({len(_val.embeddings)} embeddings / 1 RPD). {rate_limiter.status()}')
except Exception as e:
    print(f'\n❌ Batch validation FAILED: {e}')


## 4. Pipeline de extracción masiva (Optimizado con Batch)


In [ ]:
# Cargar índice del test set
with open(TEST_INDEX_PATH, 'r', encoding='utf-8') as f:
    test_index = json.load(f)

print(f'📋 Test set: {len(test_index)} vídeos')
print(f'📊 Progreso cache: {cache_progress(db)}')


In [ ]:
# ══════════════════════════════════════════════════════════════
# PIPELINE OPTIMIZADO — 2 FASES
# Fase A: ALL text embeddings via batch_embed_contents (~1 RPD per 50 texts)
# Fase B: Video embeddings one-by-one (multimodal batch not supported by API)
# Re-run safe: SQLite cache skips already-processed items automatically.
# ══════════════════════════════════════════════════════════════

print(f'📊 Cache actual: {cache_progress(db)}')
print(f'📋 {rate_limiter.status()}\n')

# ── FASE A: text embeddings ────────────────────────────────────────
print('=' * 62)
print('FASE A: Collecting pending text embeddings...')
print('=' * 62)

pending_texts = []  # (raw_id, modality, full_prefixed_text)

for item in test_index:
    raw_id = item['raw_id']
    if load_embedding(db, raw_id, 'text_real') is None:
        pending_texts.append((
            raw_id, 'text_real',
            f'task: fact checking | query: {item["title"]}'
        ))
    if load_embedding(db, raw_id, 'text_fake') is None:
        pending_texts.append((
            raw_id, 'text_fake',
            f'task: fact checking | query: {item["fake_title"]}'
        ))

print(f'  text_real + text_fake pending: {len(pending_texts)}')

# Transcripts: Whisper extraction is local (no API cost) → then batch-embed
print('\n🎧 Extracting pending transcripts with Whisper (local, no API cost)...')
transcript_errors = []
for item in test_index:
    raw_id = item['raw_id']
    if load_embedding(db, raw_id, 'transcript') is not None:
        continue
    try:
        transcript = get_transcript(item['video_path'])
        if transcript:
            pending_texts.append((
                raw_id, 'transcript',
                f'task: fact checking | document: {transcript}'
            ))
    except Exception as e:
        transcript_errors.append((raw_id, str(e)))

n_transcripts = sum(1 for _, m, _ in pending_texts if m == 'transcript')
print(f'  Transcripts ready: {n_transcripts}')
print(f'  Transcripts skipped (error): {len(transcript_errors)}')
print(f'  TOTAL texts pending: {len(pending_texts)}')
n_batches = (len(pending_texts) + 49) // 50
print(f'  Estimated RPD for Fase A: ~{n_batches} requests (batches of 50)')

if pending_texts:
    print(f'\n🚀 Running batch embed (crash-safe: saves per-batch to SQLite)...')
    # ↓ Pass db so the function saves each batch immediately after it succeeds.
    total_saved = get_text_embeddings_batch(pending_texts, db, batch_size=50)
    print(f'✅ Fase A complete: {total_saved} text embeddings saved.')
    print(f'   {rate_limiter.status()}')
else:
    print('✅ Fase A: nothing pending, all texts already cached.')

# ── FASE B: video embeddings ──────────────────────────────────────
print('\n' + '=' * 62)
print('FASE B: Individual video embeddings...')
print('=' * 62)

pending_videos = [
    item for item in test_index
    if load_embedding(db, item['raw_id'], 'video') is None
]
print(f'  Videos pending: {len(pending_videos)}')
print(f'  Estimated RPD for Fase B: ~{len(pending_videos)} requests')
print(f'  {rate_limiter.status()}')

for i, item in enumerate(pending_videos):
    raw_id     = item['raw_id']
    video_path = item['video_path']
    try:
        emb = safe_embed_fetch(get_video_embedding, video_path)
        save_embedding(db, raw_id, 'video', emb)
        print(f'\r  [{i+1}/{len(pending_videos)}] | {rate_limiter.status()}', end='', flush=True)
    except Exception as e:
        print(f'\n❌ Video error {raw_id}: {e}. Skipping...')

print(f'\n\n✅ Pipeline complete.')
print(f'📊 Final cache: {cache_progress(db)}')
print(f'📋 {rate_limiter.status()}')


## 5. Experimento E1: Baseline sim(V, T_real) vs sim(V, T_fake)


In [ ]:
results = []
skipped = 0

for item in test_index:
    raw_id = item['raw_id']
    emb_vid  = load_embedding(db, raw_id, 'video')
    emb_real = load_embedding(db, raw_id, 'text_real')
    emb_fake = load_embedding(db, raw_id, 'text_fake')

    if emb_vid is None or emb_real is None or emb_fake is None:
        skipped += 1
        continue

    sim_real = float(cosine_similarity(emb_vid.reshape(1,-1), emb_real.reshape(1,-1))[0][0])
    sim_fake = float(cosine_similarity(emb_vid.reshape(1,-1), emb_fake.reshape(1,-1))[0][0])
    sim_titles = float(cosine_similarity(emb_real.reshape(1,-1), emb_fake.reshape(1,-1))[0][0])

    correct = sim_real > sim_fake
    margin = sim_real - sim_fake

    results.append({
        'raw_id': raw_id,
        'category': item['category'],
        'sim_real': sim_real,
        'sim_fake': sim_fake,
        'sim_titles': sim_titles,
        'margin': margin,
        'correct': correct
    })

df = pd.DataFrame(results)
df.to_csv('results/E1_baseline_similarity.csv', index=False)

print(f'📊 E1 Resultados ({len(df)} vídeos, {skipped} saltados por falta de embeddings):')
print(f'   Accuracy global: {df["correct"].mean():.4f} ({df["correct"].sum()}/{len(df)})')
print(f'   Margen medio: {df["margin"].mean():.4f}')
print(f'   Margen mediano: {df["margin"].median():.4f}')
print()
print('   Accuracy por categoría:')
for cat, group in df.groupby('category'):
    print(f'     {cat:15s}: {group["correct"].mean():.4f} ({group["correct"].sum()}/{len(group)})')


## Experimento E2: Transcript como señal de verificación


In [ ]:
# === E2: Transcript-based fake detection ===
results_e2 = []
skipped_e2 = 0

for item in test_index:
    raw_id = item['raw_id']
    emb_vid   = load_embedding(db, raw_id, 'video')
    emb_real  = load_embedding(db, raw_id, 'text_real')
    emb_fake  = load_embedding(db, raw_id, 'text_fake')
    emb_trans = load_embedding(db, raw_id, 'transcript')

    if emb_vid is None or emb_real is None or emb_fake is None:
        skipped_e2 += 1
        continue

    # E1 metrics (video-based)
    sim_vid_real = float(cosine_similarity(emb_vid.reshape(1,-1), emb_real.reshape(1,-1))[0][0])
    sim_vid_fake = float(cosine_similarity(emb_vid.reshape(1,-1), emb_fake.reshape(1,-1))[0][0])
    e1_correct = sim_vid_real > sim_vid_fake
    e1_margin = sim_vid_real - sim_vid_fake

    row = {
        'raw_id': raw_id,
        'category': item['category'],
        'sim_vid_real': sim_vid_real,
        'sim_vid_fake': sim_vid_fake,
        'e1_correct': e1_correct,
        'e1_margin': e1_margin,
        'has_transcript': emb_trans is not None,
    }

    # E2 metrics (transcript-based)
    if emb_trans is not None:
        sim_trans_real = float(cosine_similarity(emb_trans.reshape(1,-1), emb_real.reshape(1,-1))[0][0])
        sim_trans_fake = float(cosine_similarity(emb_trans.reshape(1,-1), emb_fake.reshape(1,-1))[0][0])
        sim_trans_vid  = float(cosine_similarity(emb_trans.reshape(1,-1), emb_vid.reshape(1,-1))[0][0])
        row['sim_trans_real'] = sim_trans_real
        row['sim_trans_fake'] = sim_trans_fake
        row['sim_trans_vid']  = sim_trans_vid
        row['e2_correct'] = sim_trans_real > sim_trans_fake
        row['e2_margin']  = sim_trans_real - sim_trans_fake

        # E2c: Combined score (video + transcript)
        combined_real = (sim_vid_real + sim_trans_real) / 2
        combined_fake = (sim_vid_fake + sim_trans_fake) / 2
        row['e2c_correct'] = combined_real > combined_fake
        row['e2c_margin']  = combined_real - combined_fake

    results_e2.append(row)

df2 = pd.DataFrame(results_e2)
df2_trans = df2[df2['has_transcript']].copy()

print(f'📊 Resultados ({len(df2)} total, {len(df2_trans)} con transcript, {skipped_e2} saltados)')
print()

# --- E1 vs E2a vs E2c comparison ---
print('='*65)
print(f"{'Método':<30} {'Accuracy':>10} {'Margen medio':>14}")
print('='*65)
print(f"{'E1: video↔título':<30} {df2['e1_correct'].mean():>10.4f} {df2['e1_margin'].mean():>14.4f}")
if len(df2_trans) > 0:
    print(f"{'E2a: transcript↔título':<30} {df2_trans['e2_correct'].mean():>10.4f} {df2_trans['e2_margin'].mean():>14.4f}")
    print(f"{'E2c: combinado (V+T)/2':<30} {df2_trans['e2c_correct'].mean():>10.4f} {df2_trans['e2c_margin'].mean():>14.4f}")
    # E1 on same subset for fair comparison
    print(f"{'E1 (mismo subset)':<30} {df2_trans['e1_correct'].mean():>10.4f} {df2_trans['e1_margin'].mean():>14.4f}")
print('='*65)
print()

# --- By category ---
if len(df2_trans) > 0:
    print('📊 E2: Accuracy por categoría')
    print(f"{'Categoría':<15} {'E1':>8} {'E2a(trans)':>12} {'E2c(comb)':>12}")
    print('-'*50)
    for cat, g in df2_trans.groupby('category'):
        print(f"{cat:<15} {g['e1_correct'].mean():>8.4f} {g['e2_correct'].mean():>12.4f} {g['e2c_correct'].mean():>12.4f}")
    print()

df2.to_csv('results/E2_transcript_comparison.csv', index=False)
print('💾 Guardado en results/E2_transcript_comparison.csv')

In [ ]:
# === E2 Visualizaciones ===
if len(df2_trans) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('E2: Transcript vs Video para detección de fakes', fontsize=14)

    # 1. Histograma márgenes E1 vs E2a
    ax = axes[0, 0]
    ax.hist(df2_trans['e1_margin'], bins=40, alpha=0.6, label='E1: video↔título', color='steelblue')
    ax.hist(df2_trans['e2_margin'], bins=40, alpha=0.6, label='E2a: transcript↔título', color='coral')
    ax.axvline(0, color='red', linestyle='--', alpha=0.5)
    ax.set_xlabel('Margen (sim_real - sim_fake)')
    ax.set_ylabel('Frecuencia')
    ax.set_title('Distribución de márgenes')
    ax.legend()

    # 2. Scatter: margen E1 vs margen E2a
    ax = axes[0, 1]
    colors = {'person': 'blue', 'location': 'green', 'organization': 'orange', 'event': 'red'}
    for cat, g in df2_trans.groupby('category'):
        ax.scatter(g['e1_margin'], g['e2_margin'], alpha=0.4, label=cat, c=colors.get(cat, 'gray'), s=15)
    ax.axhline(0, color='red', linestyle='--', alpha=0.3)
    ax.axvline(0, color='red', linestyle='--', alpha=0.3)
    ax.set_xlabel('E1 margin (video)')
    ax.set_ylabel('E2a margin (transcript)')
    ax.set_title('E1 vs E2a por categoría')
    ax.legend(fontsize=8)

    # 3. Accuracy por categoría comparada
    ax = axes[1, 0]
    cats = sorted(df2_trans['category'].unique())
    x = range(len(cats))
    w = 0.25
    acc_e1 = [df2_trans[df2_trans['category']==c]['e1_correct'].mean() for c in cats]
    acc_e2 = [df2_trans[df2_trans['category']==c]['e2_correct'].mean() for c in cats]
    acc_e2c = [df2_trans[df2_trans['category']==c]['e2c_correct'].mean() for c in cats]
    ax.bar([i-w for i in x], acc_e1, w, label='E1: video', color='steelblue')
    ax.bar(list(x), acc_e2, w, label='E2a: transcript', color='coral')
    ax.bar([i+w for i in x], acc_e2c, w, label='E2c: combinado', color='mediumseagreen')
    ax.set_xticks(list(x))
    ax.set_xticklabels(cats, rotation=15)
    ax.set_ylabel('Accuracy')
    ax.set_title('Accuracy por categoría y método')
    ax.legend(fontsize=8)
    ax.set_ylim(0.5, 1.05)

    # 4. Alineamiento multimodal: sim(transcript, video)
    ax = axes[1, 1]
    ax.hist(df2_trans['sim_trans_vid'], bins=40, color='mediumpurple', alpha=0.7)
    ax.set_xlabel('sim(transcript, vídeo)')
    ax.set_ylabel('Frecuencia')
    ax.set_title('Alineamiento transcript↔vídeo')
    ax.axvline(df2_trans['sim_trans_vid'].mean(), color='red', linestyle='--',
               label=f'media={df2_trans["sim_trans_vid"].mean():.3f}')
    ax.legend()

    plt.tight_layout()
    plt.savefig('results/figures/E2_transcript_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('📊 Guardado en results/figures/E2_transcript_analysis.png')

## Experimento E2d: Análisis de alineamiento multimodal


In [ ]:
# === E2d: Multimodal alignment triangle ===
if len(df2_trans) > 0:
    # Compute alignment scores
    # For real title: alignment = mean(sim_vid_real, sim_trans_real, sim_trans_vid)
    # For fake title: alignment = mean(sim_vid_fake, sim_trans_fake, sim_trans_vid)
    df2_trans = df2_trans.copy()
    df2_trans['align_real'] = (df2_trans['sim_vid_real'] + df2_trans['sim_trans_real'] + df2_trans['sim_trans_vid']) / 3
    df2_trans['align_fake'] = (df2_trans['sim_vid_fake'] + df2_trans['sim_trans_fake'] + df2_trans['sim_trans_vid']) / 3
    df2_trans['align_diff'] = df2_trans['align_real'] - df2_trans['align_fake']
    df2_trans['e2d_correct'] = df2_trans['align_real'] > df2_trans['align_fake']

    print('='*65)
    print('📐 E2d: Alineamiento multimodal (triángulo V-T-Transcript)')
    print('='*65)
    print(f"Accuracy (alineamiento real > fake): {df2_trans['e2d_correct'].mean():.4f}")
    print(f"Diferencia media de alineamiento:    {df2_trans['align_diff'].mean():.4f}")
    print()

    # Comparison table
    print(f"{'Método':<35} {'Accuracy':>10}")
    print('-'*47)
    print(f"{'E1:  video ↔ título':<35} {df2_trans['e1_correct'].mean():>10.4f}")
    print(f"{'E2a: transcript ↔ título':<35} {df2_trans['e2_correct'].mean():>10.4f}")
    print(f"{'E2c: (video+transcript)/2 ↔ título':<35} {df2_trans['e2c_correct'].mean():>10.4f}")
    print(f"{'E2d: alineamiento triángulo V-T-Tr':<35} {df2_trans['e2d_correct'].mean():>10.4f}")
    print()

    # Cases where E2a corrects E1 errors
    e1_wrong_e2_right = df2_trans[(~df2_trans['e1_correct']) & (df2_trans['e2_correct'])]
    e1_right_e2_wrong = df2_trans[(df2_trans['e1_correct']) & (~df2_trans['e2_correct'])]
    both_wrong = df2_trans[(~df2_trans['e1_correct']) & (~df2_trans['e2_correct'])]
    both_right = df2_trans[(df2_trans['e1_correct']) & (df2_trans['e2_correct'])]

    print('📊 Complementariedad E1 vs E2a:')
    print(f'   Ambos correctos:       {len(both_right):>4} ({len(both_right)/len(df2_trans):.1%})')
    print(f'   E1 correcto, E2a falla: {len(e1_right_e2_wrong):>4} ({len(e1_right_e2_wrong)/len(df2_trans):.1%})')
    print(f'   E1 falla, E2a correcto: {len(e1_wrong_e2_right):>4} ({len(e1_wrong_e2_right)/len(df2_trans):.1%})')
    print(f'   Ambos fallan:           {len(both_wrong):>4} ({len(both_wrong)/len(df2_trans):.1%})')
    print()
    print(f'   → El transcript rescata {len(e1_wrong_e2_right)} casos que el vídeo solo no detecta.')
    print(f'   → Union (oracle): {(len(both_right) + len(e1_wrong_e2_right) + len(e1_right_e2_wrong))/len(df2_trans):.4f}')

## 6. Experimento E4a: Fragilidad por categoría (datos reales)


In [ ]:
if len(df) > 0:
    print('📊 E4a: Similitud entre título real y fake (cuanto más alta, más difícil de distinguir):')
    print()
    for cat, group in df.groupby('category'):
        acc = group['correct'].mean()
        sim = group['sim_titles'].mean()
        print(f'  {cat:15s}: sim_títulos media = {sim:.4f}  |  accuracy E1 = {acc:.4f}  |  n = {len(group)}')

    print()
    corr = df[['sim_titles', 'margin']].corr().iloc[0,1]
    print(f'  Correlación (sim_títulos vs margen E1): {corr:.4f}')
    print('  → Si es negativa: pares con títulos más similares tienen menor margen (más difíciles).')


## 7. Visualizaciones


In [ ]:
if len(df) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # 1. Histograma de márgenes
    ax = axes[0, 0]
    ax.hist(df['margin'], bins=50, edgecolor='black', alpha=0.7)
    ax.axvline(0, color='red', linestyle='--', label='Umbral (margen=0)')
    ax.set_xlabel('Margen (sim_real - sim_fake)')
    ax.set_ylabel('Frecuencia')
    ax.set_title('E1: Distribución de márgenes')
    ax.legend()

    # 2. Scatter sim_real vs sim_fake coloreado por categoría
    ax = axes[0, 1]
    colors = {'person': 'tab:blue', 'location': 'tab:orange', 'event': 'tab:green', 'organization': 'tab:red'}
    for cat, group in df.groupby('category'):
        ax.scatter(group['sim_real'], group['sim_fake'], alpha=0.4, s=10,
                   color=colors.get(cat, 'gray'), label=cat)
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
    ax.set_xlabel('sim(video, título_real)')
    ax.set_ylabel('sim(video, título_fake)')
    ax.set_title('E1: sim_real vs sim_fake')
    ax.legend()

    # 3. Box plot de márgenes por categoría
    ax = axes[1, 0]
    categories = ['person', 'location', 'event', 'organization']
    data_by_cat = [df[df['category'] == c]['margin'].values for c in categories]
    ax.boxplot(data_by_cat, labels=categories)
    ax.axhline(0, color='red', linestyle='--', alpha=0.5)
    ax.set_ylabel('Margen')
    ax.set_title('E1: Margen por categoría')

    # 4. Accuracy por categoría (barras)
    ax = axes[1, 1]
    acc_by_cat = df.groupby('category')['correct'].mean()
    acc_by_cat = acc_by_cat.reindex(categories)
    bars = ax.bar(categories, acc_by_cat.values, color=[colors[c] for c in categories])
    ax.set_ylabel('Accuracy')
    ax.set_title('E1: Accuracy por categoría')
    ax.set_ylim(0, 1)
    for bar, val in zip(bars, acc_by_cat.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.2%}', ha='center', va='bottom', fontsize=9)

    plt.tight_layout()
    plt.savefig('results/figures/E1_overview.png', dpi=150)
    plt.show()
    print('📊 Guardado en results/figures/E1_overview.png')


In [ ]:
if len(df) > 0:
    fig, ax = plt.subplots(figsize=(8, 5))
    for cat, group in df.groupby('category'):
        ax.scatter(group['sim_titles'], group['margin'], alpha=0.4, s=10,
                   color=colors.get(cat, 'gray'), label=cat)
    ax.axhline(0, color='red', linestyle='--', alpha=0.5)
    ax.set_xlabel('sim(título_real, título_fake) — dificultad del par')
    ax.set_ylabel('Margen E1 (sim_real - sim_fake)')
    ax.set_title('E4a: ¿Los pares más similares son más difíciles?')
    ax.legend()
    plt.tight_layout()
    plt.savefig('results/figures/E4a_fragility.png', dpi=150)
    plt.show()
    print('📊 Guardado en results/figures/E4a_fragility.png')


## 8. Resumen de errores (vídeos donde E1 falla)


In [ ]:
if len(df) > 0:
    errors = df[~df['correct']].sort_values('margin')
    print(f'❌ Vídeos donde E1 falla: {len(errors)}/{len(df)} ({len(errors)/len(df):.1%})')
    print(f'   Por categoría: {errors.groupby("category").size().to_dict()}')
    print()

    if len(errors) > 0:
        meta = {}
        with open('annotations/caption_metadata.jsonl', 'r', encoding='utf-8') as f:
            for line in f:
                obj = json.loads(line)
                meta[obj['raw_id']] = obj

        print('   Top 10 peores errores:')
        for _, row in errors.head(10).iterrows():
            m = meta.get(row['raw_id'], {})
            print(f'   [{row["category"]}] margen={row["margin"]:.4f}')
            print(f'     Real: {m.get("title", "?")[:80]}')
            print(f'     Fake: {m.get("fake_title", "?")[:80]}')
            print()


## Prueba rápida con un solo vídeo (test manual)


In [ ]:
# Prueba con el vídeo del presidente serbio (ejemplo original)
test_id = '00a6e270c1008f8e3629703a59094497'
test_video_path = f'test_video/{test_id}.mp4'

if os.path.exists(test_video_path):
    emb_vid = get_or_compute(db, test_id, 'video',
        lambda: safe_embed_fetch(get_video_embedding, test_video_path))
    emb_real = get_or_compute(db, test_id, 'text_real',
        lambda: safe_embed_fetch(get_text_embedding,
            "Serbian president not interested in 'most corrupt and deceitful' Guardian coverage."))
    emb_fake = get_or_compute(db, test_id, 'text_fake',
        lambda: safe_embed_fetch(get_text_embedding,
            "Serbian president not interested in 'most corrupt and deceitful' BBC coverage."))

    sim_real = float(cosine_similarity(emb_vid.reshape(1,-1), emb_real.reshape(1,-1))[0][0])
    sim_fake = float(cosine_similarity(emb_vid.reshape(1,-1), emb_fake.reshape(1,-1))[0][0])

    print(f'sim(video, título_real): {sim_real:.4f}')
    print(f'sim(video, título_fake): {sim_fake:.4f}')
    print(f'Margen: {sim_real - sim_fake:.4f}')
    print(f'✅ Correcto: {sim_real > sim_fake}')
else:
    print(f'⚠️ Vídeo {test_video_path} no encontrado.')
